[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_02/SFM_ch2_risk_management/SFM_ch2_risk_management.ipynb)

# SFM_ch2_risk_management

Risk Measures: VaR, Expected Shortfall, and Backtesting
Description:
- VaR and ES at 95%, 99%, 99.9% confidence levels
- Comparison under Normal, Student-t, and Empirical distributions
- Rolling 250-day VaR backtest
- Basel traffic light test for model validation
- Portfolio impact analysis ($1M portfolio)
Statistics of Financial Markets course

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yfinance as yf
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Chart style settings - Nature journal quality
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 0.75

# Color palette
MAIN_BLUE = '#1A3A6E'
CRIMSON = '#DC3545'
FOREST = '#2E7D32'
AMBER = '#B5853F'
ORANGE = '#E67E22'

# Output directory
CHART_DIR = os.path.join('..', '..', '..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(name):
    """Save figure with transparent background."""
    plt.savefig(os.path.join(CHART_DIR, f'{name}.pdf'),
                bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.png'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.close()
    print(f"   Saved: {name}.pdf/.png")

## Download Data

In [ ]:
# Download S&P 500 data
data = yf.download('^GSPC', start='2000-01-01', end='2025-01-01', progress=False)
close = data['Close'].squeeze()
log_ret = np.log(close / close.shift(1)).dropna()

print(f"   Period: {close.index[0].strftime('%Y-%m-%d')} to {close.index[-1].strftime('%Y-%m-%d')}")
print(f"   Observations: {len(log_ret)}")
print(f"   Mean:  {log_ret.mean():.6f}")
print(f"   Std:   {log_ret.std():.6f}")
print(f"   Skew:  {log_ret.skew():.4f}")
print(f"   Kurt:  {log_ret.kurtosis():.4f}")
print(f"   Min:   {log_ret.min():.6f}")
print(f"   Max:   {log_ret.max():.6f}")

## VaR and ES Comparison

In [ ]:
# --- Fit distributions to full sample ---
mu_n, sigma_n = stats.norm.fit(log_ret)
t_df, t_loc, t_scale = stats.t.fit(log_ret)

print(f"   Normal  MLE:  mu = {mu_n:.6f},  sigma = {sigma_n:.6f}")
print(f"   Student-t MLE:  df = {t_df:.2f},  loc = {t_loc:.6f},  scale = {t_scale:.6f}")
print()

# --- Confidence levels ---
conf_levels = [0.95, 0.99, 0.999]
portfolio_value = 1_000_000  # $1M

rows = []

for conf in conf_levels:
    alpha = 1 - conf

    # ---- Normal VaR and ES ----
    z_alpha = stats.norm.ppf(alpha)
    var_norm = -(mu_n + sigma_n * z_alpha)
    # ES_Normal = mu - sigma * phi(z_alpha) / (1 - conf)
    es_norm = -(mu_n - sigma_n * stats.norm.pdf(z_alpha) / alpha)

    # ---- Student-t VaR and ES ----
    t_q = stats.t.ppf(alpha, t_df, loc=0, scale=1)
    var_t = -(t_loc + t_scale * t_q)
    # ES_t = loc - (df + t_q^2)/(df-1) * f(t_q)/(1-conf) * scale
    f_t_q = stats.t.pdf(t_q, t_df)
    es_t = -(t_loc - t_scale * (t_df + t_q**2) / (t_df - 1) * f_t_q / alpha)

    # ---- Empirical VaR and ES ----
    var_emp = -np.quantile(log_ret, alpha)
    tail_returns = log_ret[log_ret <= -var_emp]
    es_emp = -tail_returns.mean()

    rows.append({
        'Confidence': f"{conf:.1%}",
        'VaR Normal': var_norm,
        'VaR Student-t': var_t,
        'VaR Empirical': var_emp,
        'ES Normal': es_norm,
        'ES Student-t': es_t,
        'ES Empirical': es_emp,
    })

df_table = pd.DataFrame(rows).set_index('Confidence')

# --- Print percentage table ---
print("   === VaR and ES (as % of portfolio) ===")
print()
header = f"   {'Conf':>7}  {'VaR-N':>8}  {'VaR-t':>8}  {'VaR-Emp':>8}  {'ES-N':>8}  {'ES-t':>8}  {'ES-Emp':>8}"
print(header)
print(f"   {'-' * len(header.strip())}")
for _, row in df_table.iterrows():
    print(f"   {row.name:>7}  {row['VaR Normal']:>8.4%}  {row['VaR Student-t']:>8.4%}  "
          f"{row['VaR Empirical']:>8.4%}  {row['ES Normal']:>8.4%}  {row['ES Student-t']:>8.4%}  "
          f"{row['ES Empirical']:>8.4%}")
print()

# --- Print dollar amounts for $1M portfolio ---
print(f"   === Dollar Impact on ${portfolio_value:,.0f} Portfolio ===")
print()
header2 = f"   {'Conf':>7}  {'VaR-N':>10}  {'VaR-t':>10}  {'VaR-Emp':>10}  {'ES-N':>10}  {'ES-t':>10}  {'ES-Emp':>10}"
print(header2)
print(f"   {'-' * len(header2.strip())}")
for _, row in df_table.iterrows():
    print(f"   {row.name:>7}  ${row['VaR Normal'] * portfolio_value:>9,.0f}  "
          f"${row['VaR Student-t'] * portfolio_value:>9,.0f}  "
          f"${row['VaR Empirical'] * portfolio_value:>9,.0f}  "
          f"${row['ES Normal'] * portfolio_value:>9,.0f}  "
          f"${row['ES Student-t'] * portfolio_value:>9,.0f}  "
          f"${row['ES Empirical'] * portfolio_value:>9,.0f}")
print()

# --- Styled matplotlib table figure ---
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.axis('off')

col_labels = ['Confidence', 'VaR\nNormal', 'VaR\nStudent-t', 'VaR\nEmpirical',
              'ES\nNormal', 'ES\nStudent-t', 'ES\nEmpirical']

cell_text = []
for _, row in df_table.iterrows():
    cell_text.append([
        row.name,
        f"${row['VaR Normal'] * portfolio_value:,.0f}\n({row['VaR Normal']:.3%})",
        f"${row['VaR Student-t'] * portfolio_value:,.0f}\n({row['VaR Student-t']:.3%})",
        f"${row['VaR Empirical'] * portfolio_value:,.0f}\n({row['VaR Empirical']:.3%})",
        f"${row['ES Normal'] * portfolio_value:,.0f}\n({row['ES Normal']:.3%})",
        f"${row['ES Student-t'] * portfolio_value:,.0f}\n({row['ES Student-t']:.3%})",
        f"${row['ES Empirical'] * portfolio_value:,.0f}\n({row['ES Empirical']:.3%})",
    ])

table = ax.table(cellText=cell_text, colLabels=col_labels,
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(7.5)
table.scale(1.0, 2.2)

# Style header row
for j in range(len(col_labels)):
    cell = table[0, j]
    cell.set_facecolor(MAIN_BLUE)
    cell.set_text_props(color='white', fontweight='bold')
    cell.set_edgecolor('white')

# Style data rows
row_colors = ['#F0F4FA', '#FFFFFF', '#F0F4FA']
for i in range(len(cell_text)):
    for j in range(len(col_labels)):
        cell = table[i + 1, j]
        cell.set_facecolor(row_colors[i])
        cell.set_edgecolor('#CCCCCC')
        if j == 0:
            cell.set_text_props(fontweight='bold')

ax.set_title('VaR and ES: $1M Portfolio Impact', fontsize=10, fontweight='bold',
             pad=15, color=MAIN_BLUE)

plt.tight_layout()
save_fig('ch2_risk_comparison_table')

## Rolling VaR Backtest

In [ ]:
# --- Rolling 250-day VaR backtest at 99% confidence ---
window = 250
conf_bt = 0.99
alpha_bt = 1 - conf_bt

returns = log_ret.values
dates = log_ret.index

var_normal_roll = np.full(len(returns), np.nan)
var_t_roll = np.full(len(returns), np.nan)

for i in range(window, len(returns)):
    window_rets = returns[i - window:i]

    # Normal VaR
    mu_w, sigma_w = stats.norm.fit(window_rets)
    var_normal_roll[i] = -(mu_w + sigma_w * stats.norm.ppf(alpha_bt))

    # Student-t VaR
    try:
        df_w, loc_w, scale_w = stats.t.fit(window_rets)
        var_t_roll[i] = -(loc_w + scale_w * stats.t.ppf(alpha_bt, df_w))
    except Exception:
        var_t_roll[i] = var_normal_roll[i]

# Identify exceedances (actual loss exceeds VaR)
valid = ~np.isnan(var_normal_roll)
exc_normal = valid & (-returns > var_normal_roll)
exc_t = valid & (-returns > var_t_roll)

n_valid = np.sum(valid)
n_exc_normal = np.sum(exc_normal)
n_exc_t = np.sum(exc_t)
expected_exc = n_valid * alpha_bt

print(f"   Rolling window: {window} days")
print(f"   Confidence level: {conf_bt:.0%}")
print(f"   Test period days: {n_valid}")
print(f"   Expected exceedances: {expected_exc:.1f}")
print(f"   Normal VaR exceedances: {n_exc_normal}")
print(f"   Student-t VaR exceedances: {n_exc_t}")
print()

# --- Two-panel figure ---
fig = plt.figure(figsize=(9, 6))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1.5], hspace=0.35)

# Panel A: Returns with VaR lines and exceedances
ax1 = fig.add_subplot(gs[0])

ax1.plot(dates, returns, color='#AAAAAA', linewidth=0.3, alpha=0.6, label='Log-returns')
ax1.plot(dates[valid], -var_normal_roll[valid], color=MAIN_BLUE, linewidth=0.7,
         label=f'Normal VaR {conf_bt:.0%}')
ax1.plot(dates[valid], -var_t_roll[valid], color=FOREST, linewidth=0.7,
         linestyle='--', label=f'Student-t VaR {conf_bt:.0%}')

# Mark exceedances with red triangles
ax1.scatter(dates[exc_normal], returns[exc_normal], color=CRIMSON, s=12,
            marker='v', zorder=5, label=f'Exceedances (N={n_exc_normal})')

ax1.set_xlabel('Date')
ax1.set_ylabel('Log-return')
ax1.set_title('Panel A: Rolling 99% VaR Backtest', fontsize=9, loc='left')
ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=4,
           frameon=False, fontsize=7)

# Panel B: Cumulative exceedance count vs expected
ax2 = fig.add_subplot(gs[1])

cum_exc_normal = np.cumsum(exc_normal[valid])
cum_exc_t = np.cumsum(exc_t[valid])
expected_line = np.arange(1, n_valid + 1) * alpha_bt

ax2.plot(dates[valid], cum_exc_normal, color=MAIN_BLUE, linewidth=0.8,
         label='Normal exceedances')
ax2.plot(dates[valid], cum_exc_t, color=FOREST, linewidth=0.8,
         linestyle='--', label='Student-t exceedances')
ax2.plot(dates[valid], expected_line, color=CRIMSON, linewidth=0.7,
         linestyle=':', label='Expected (1%)')

ax2.set_xlabel('Date')
ax2.set_ylabel('Cumulative Exceedances')
ax2.set_title('Panel B: Cumulative Exceedance Count', fontsize=9, loc='left')
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3,
           frameon=False, fontsize=7)

save_fig('ch2_var_backtest')

## Basel Traffic Light Test

In [ ]:
# --- Basel traffic light test on last 250 trading days ---
test_window = 250
expected_exc_250 = test_window * alpha_bt  # 250 * 0.01 = 2.5

# Count exceedances in the last 250 days
exc_normal_last = np.sum(exc_normal[-test_window:])
exc_t_last = np.sum(exc_t[-test_window:])

def basel_zone(n_exc):
    """Basel traffic light classification."""
    if n_exc < 5:
        return 'GREEN', '#2E7D32'
    elif n_exc <= 9:
        return 'YELLOW', '#B5853F'
    else:
        return 'RED', '#DC3545'

zone_normal, color_normal = basel_zone(exc_normal_last)
zone_t, color_t = basel_zone(exc_t_last)

print(f"   === Basel Traffic Light Test ===")
print(f"   Test window: last {test_window} trading days")
print(f"   VaR confidence: {conf_bt:.0%}")
print(f"   Expected exceedances: {expected_exc_250:.1f}")
print()
print(f"   Normal VaR:    {exc_normal_last} exceedances  =>  {zone_normal} zone")
print(f"   Student-t VaR: {exc_t_last} exceedances  =>  {zone_t} zone")
print()
print(f"   Basel zones:")
print(f"     GREEN  : 0-4 exceedances  (model accepted)")
print(f"     YELLOW : 5-9 exceedances  (increased capital multiplier)")
print(f"     RED    : 10+ exceedances  (model rejected)")
print()

# --- Visual indicator ---
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.axis('off')

# Draw traffic light zones as horizontal bands
zone_ranges = [(0, 4, 'GREEN\n(Accepted)', FOREST, 0.2),
               (5, 9, 'YELLOW\n(Higher Capital)', AMBER, 0.2),
               (10, 15, 'RED\n(Rejected)', CRIMSON, 0.2)]

y_positions = [0.72, 0.45, 0.18]
bar_height = 0.18

for (lo, hi, label, color, alpha_val), y in zip(zone_ranges, y_positions):
    ax.barh(y, hi - lo + 1, left=lo, height=bar_height,
            color=color, alpha=0.25, edgecolor=color, linewidth=1.0)
    ax.text(-1.5, y, label, ha='right', va='center', fontsize=8,
            fontweight='bold', color=color)
    # Zone boundary labels
    ax.text(lo, y - bar_height / 2 - 0.03, str(lo), ha='center',
            va='top', fontsize=7, color='gray')
    ax.text(hi, y - bar_height / 2 - 0.03, str(hi), ha='center',
            va='top', fontsize=7, color='gray')

# Plot exceedance markers
ax.plot(exc_normal_last, 0.72, 'o', markersize=10, color=MAIN_BLUE,
        markeredgecolor='white', markeredgewidth=1.5, zorder=5)
ax.text(exc_normal_last, 0.72 + bar_height / 2 + 0.04,
        f'Normal\n({exc_normal_last} exc.)', ha='center', va='bottom',
        fontsize=7, color=MAIN_BLUE, fontweight='bold')

ax.plot(exc_t_last, 0.45, 's', markersize=10, color=FOREST,
        markeredgecolor='white', markeredgewidth=1.5, zorder=5)
ax.text(exc_t_last, 0.45 + bar_height / 2 + 0.04,
        f'Student-t\n({exc_t_last} exc.)', ha='center', va='bottom',
        fontsize=7, color=FOREST, fontweight='bold')

# Expected exceedances reference line
ax.axvline(x=expected_exc_250, color=CRIMSON, linestyle=':', linewidth=0.8)
ax.text(expected_exc_250, 0.05, f'Expected\n({expected_exc_250:.1f})',
        ha='center', va='top', fontsize=7, color=CRIMSON)

ax.set_xlim(-5, 16)
ax.set_ylim(0, 1)
ax.set_title('Basel Traffic Light Test (last 250 days, 99% VaR)',
             fontsize=10, fontweight='bold', color=MAIN_BLUE, pad=10)

plt.tight_layout()
save_fig('ch2_basel_traffic_light')